# Day 193 — Final Capstone Part 2: Data Layer

**Month 12 · Day 2 of Final Capstone (Days 192–196)**

Day 192 profiled the raw TeleServe India dataset and locked the 5-layer architecture. Today builds
**Layer 2 (Transformation/Cleaning)** — the only layer everything else depends on. The ML layer,
the RAG layer, and the Streamlit app all read from this layer's output, never from raw data directly.

**What Day 192 found (carried in as fixed targets, not re-profiled from memory):**
| Issue | Day 192 finding |
|---|---|
| Duplicate rows | 9 injected duplicates |
| `plan_type` casing | 6 raw values for a 3-value field (`Prepaid`/`Postpaid`/`Broadband` + 3 dirty casings) |
| `satisfaction_score` missing | Missing exactly where `resolved == False` — **structurally missing (MNAR), not random** |

**Corrected Day 192 stance on `satisfaction_score` (do not re-litigate today):** it is **left as
NaN, never imputed**. Imputing it would fabricate a satisfaction opinion for a customer whose
complaint was never resolved — there is no signal to impute from. The cleaning pipeline must
preserve this, not "fix" it away.


## Concept Notes — Why a Pipeline Class, Not a Sequence of Cells

Month 11 (Days 181–182, 189) already established the pattern: a raw-data-in, clean-data-out class
with named methods and a `run()` orchestrator, gated by an assertion test suite before anything
downstream is allowed to trust the output.

Today's `TeleServeCleaningPipeline` follows the exact same shape, because **this is the pattern a
client pays for** — not one-off cleaning code that only works once, but a class that can be re-run
every time new TeleServe records arrive, with tests that fail loudly instead of silently passing
bad data forward.

**The three cleaning operations, in dependency order:**
1. **Normalize `plan_type` casing** — must happen before anything groups or filters by plan, or
   `"Prepaid"` and `"prepaid"` silently become two different segments.
2. **Drop duplicate rows** — must happen before any aggregate (revenue, counts) is computed, or
   every duplicated customer is double-counted.
3. **Leave `satisfaction_score` as NaN where missing** — this is a *decision to preserve*, not an
   operation to perform. The test suite checks that nothing downstream accidentally fills it.

**Order matters:** normalize casing → drop duplicates → verify satisfaction_score untouched.
Dropping duplicates first, before casing is normalized, risks missing duplicate rows that only
differ by `plan_type` casing (e.g. a `"Prepaid"` row and a `"prepaid"` copy of it) — those look
like distinct rows to `.duplicated()` until casing is normalized first.


---
## Section 1 — Raw Data (Locked, DO NOT MODIFY)

Identical generator to Day 192, seed=192. This cell is locked for the rest of the capstone arc
(Days 192–196) — every day re-reads this same raw dataset, never a re-randomized version.


In [1]:
# ============================================================
# RAW DATA GENERATOR — TeleServe India (seed=192) — DO NOT MODIFY
# Locked for the entire Month 12 Final Capstone arc (Days 192-196)
# Identical to Day 192 - do not alter, or downstream numbers stop matching the answer key
# ============================================================
import pandas as pd
import numpy as np
import random

SEED = 192
random.seed(SEED)
np.random.seed(SEED)

N = 400

plan_types_clean = ["Prepaid", "Postpaid", "Broadband"]
# a few rows get inconsistent casing on purpose - today's cleaning target
plan_types_dirty = ["prepaid", "POSTPAID", "Broadband "]

complaint_templates = [
    "Network signal keeps dropping in {area} area since last week",
    "Billed amount does not match my selected {plan} plan this month",
    "Internet speed far below promised {plan} plan speed during evening hours",
    "Recharge done but plan benefits not activated for 2 days",
    "Customer care call disconnected three times without resolving {issue}",
    "Data balance deducted incorrectly after using app for {issue}",
    "Broadband router keeps restarting every few hours in {area}",
    "Requested plan downgrade last month but still billed for old plan",
    "SMS service not working since the last recharge cycle",
    "Roaming charges applied despite being within home {area} circle",
]
areas = ["Sonipat", "Gurugram", "Panipat", "Rohtak", "Karnal", "Delhi NCR"]
issues = ["video streaming", "UPI payment", "OTP delivery", "customer app login"]

rows = []
for i in range(N):
    cust_id = f"TS{1000+i}"
    plan = random.choice(plan_types_clean)
    tenure = int(np.clip(np.random.exponential(scale=18), 1, 96))
    if plan == "Prepaid":
        bill = round(np.random.normal(249, 60), 2)
    elif plan == "Postpaid":
        bill = round(np.random.normal(599, 120), 2)
    else:
        bill = round(np.random.normal(999, 180), 2)
    bill = max(bill, 99)

    template = random.choice(complaint_templates)
    complaint = template.format(area=random.choice(areas), plan=plan,
                                 issue=random.choice(issues))

    resolved = random.random() < 0.72
    days_to_resolve = int(np.clip(np.random.exponential(scale=2.5), 0, 14)) if resolved else np.nan
    satisfaction = None
    if resolved:
        base = 5 - (days_to_resolve / 4)
        satisfaction = int(np.clip(round(base + np.random.normal(0, 0.6)), 1, 5))

    signup_month_offset = tenure
    signup_date = pd.Timestamp("2026-07-06") - pd.DateOffset(months=int(signup_month_offset))

    rows.append({
        "customer_id": cust_id,
        "plan_type": plan,
        "monthly_bill_inr": bill,
        "tenure_months": tenure,
        "signup_date": signup_date.strftime("%Y-%m-%d"),
        "complaint_text": complaint,
        "resolved": resolved,
        "days_to_resolve": days_to_resolve,
        "satisfaction_score": satisfaction,
    })

df_raw = pd.DataFrame(rows)

# inject realistic data-quality issues (Day 192 profiled these, Day 193 fixes them)
dirty_idx = np.random.choice(df_raw.index, size=14, replace=False)
for j, idx in enumerate(dirty_idx):
    df_raw.loc[idx, "plan_type"] = plan_types_dirty[j % len(plan_types_dirty)]

dup_rows = df_raw.sample(n=9, random_state=SEED)
df_raw = pd.concat([df_raw, dup_rows], ignore_index=True)

print(f"TeleServe India raw dataset loaded: {len(df_raw)} rows, {df_raw.shape[1]} columns")
df_raw.head()


TeleServe India raw dataset loaded: 409 rows, 9 columns


,customer_id,plan_type,monthly_bill_inr,tenure_months,signup_date,complaint_text,resolved,days_to_resolve,satisfaction_score
0,TS1000,Postpaid,469.76,25,2024-06-06,Customer care call disconnected three times wi...,True,6.0,2.0
1,TS1001,Postpaid,376.93,1,2026-06-06,Data balance deducted incorrectly after using ...,True,0.0,5.0
2,TS1002,Postpaid,653.68,19,2024-12-06,Internet speed far below promised Postpaid pla...,True,3.0,4.0
3,TS1003,Prepaid,205.16,21,2024-10-06,Customer care call disconnected three times wi...,False,NaN,NaN
4,TS1004,Postpaid,495.01,62,2021-05-06,Data balance deducted incorrectly after using ...,True,3.0,4.0


---
## Section 2 — Practice Tasks

### Task 1 — `TeleServeCleaningPipeline` Class (30 pts)
Build the cleaning pipeline as a class, matching the Month 11 pattern (`extract` → transform steps
→ `run_tests` → `run`). Each method does exactly one job so the test suite can check them in
isolation.


In [2]:
# TASK 1: Build the cleaning pipeline class
# Goal: create a reusable pipeline that normalizes casing, removes duplicates,
#       and verifies that satisfaction_score remains structurally missing (MNAR).
# Method: one method per operation, with a run() orchestrator and a test gate.

class TeleServeCleaningPipeline:
    def __init__(self, raw_df):
        self.raw_df = raw_df.copy()
        self.df = None

    def load(self):
        """Start the pipeline from a copy of raw data. Never mutate self.raw_df."""
        self.df = self.raw_df.copy()
        print("Loaded raw data into pipeline.")

    def normalize_plan_type(self):
        """Collapse all casing/whitespace variants of plan_type down to the 3 canonical values."""
        # Strip whitespace and capitalize first letter, lower rest → "Prepaid", "Postpaid", "Broadband"
        self.df["plan_type"] = self.df["plan_type"].str.strip().str.capitalize()
        print("Normalized plan_type casing.")

    def drop_duplicates(self):
        """Remove exact duplicate rows. Must run AFTER normalize_plan_type."""
        initial_len = len(self.df)
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        removed = initial_len - len(self.df)
        print(f"Dropped {removed} duplicate rows.")

    def verify_satisfaction_untouched(self):
        """
        Not a transformation - a guard. Confirms satisfaction_score is still NaN
        wherever resolved == False, and raises if anything has silently imputed it.
        """
        mismatched = self.df[(self.df["resolved"] == False) & (self.df["satisfaction_score"].notna())]
        if len(mismatched) > 0:
            raise AssertionError(
                f"satisfaction_score was imputed where it should stay NaN: {len(mismatched)} rows"
            )
        print("Verified: satisfaction_score remains NaN for all unresolved complaints (MNAR preserved).")

    def run_tests(self):
        """
        Assertion gate - must pass before run() returns cleaned data.
        Print PASS/FAIL per check, do not just assert silently.
        """
        checks = {
            "plan_type has exactly 3 unique values": self.df["plan_type"].nunique() == 3,
            "No duplicate rows": self.df.duplicated().sum() == 0,
            "Total rows == 400 (409 raw - 9 duplicates)": len(self.df) == 400,
            "No imputation on unresolved (resolved==False)": 
                not ((self.df["resolved"] == False) & (self.df["satisfaction_score"].notna())).any(),
            "All resolved complaints have a satisfaction score": 
                not ((self.df["resolved"] == True) & (self.df["satisfaction_score"].isna())).any(),
        }

        all_pass = True
        for desc, condition in checks.items():
            if condition:
                print(f"PASS: {desc}")
            else:
                print(f"FAIL: {desc}")
                all_pass = False

        if not all_pass:
            raise AssertionError("Test suite failed – cleaning output is not valid for downstream use.")
        else:
            print("All tests passed – data layer is trustworthy.")

    def run(self):
        """One-call orchestration: load -> normalize -> dedupe -> verify -> test gate -> return."""
        self.load()
        self.normalize_plan_type()
        self.drop_duplicates()
        self.verify_satisfaction_untouched()
        self.run_tests()
        return self.df

### Task 2 — Assertion Test Suite (20 pts)
Write these as explicit checks inside `run_tests()` — each one prints `PASS`/`FAIL` with the
actual value, not a silent `assert`. This is the same discipline as Day 189's 6-assertion gate.

Required checks:
1. `df["plan_type"].nunique() == 3`
2. `df.duplicated().sum() == 0`
3. `len(df) == 400` (409 raw − 9 duplicates)
4. No row has `resolved == False` and `satisfaction_score` not null
5. No row has `resolved == True` and `satisfaction_score` null (a resolved complaint should always
   have gotten a score in this generator — if this fails, something upstream broke, not just a
   data-quality quirk to shrug off)


In [3]:
# TASK 2: Run the pipeline and show the test suite output
# Goal: execute the pipeline and print the results of all assertions.
# Method: instantiate the class with df_raw, call .run(), and display key metrics.

pipeline = TeleServeCleaningPipeline(df_raw)
df_clean = pipeline.run()

print("\n--- Cleaned data summary ---")
print(f"Shape: {df_clean.shape}")
print(f"Unique plan_type values: {sorted(df_clean['plan_type'].unique())}")
print(f"Duplicate rows remaining: {df_clean.duplicated().sum()}")
print(f"Missing satisfaction_score (unresolved): {df_clean['satisfaction_score'].isna().sum()}")
print(f"Rows with resolved==False & satisfaction not null: {((df_clean['resolved']==False) & (df_clean['satisfaction_score'].notna())).sum()}")
print(f"Rows with resolved==True & satisfaction null: {((df_clean['resolved']==True) & (df_clean['satisfaction_score'].isna())).sum()}")

Loaded raw data into pipeline.
Normalized plan_type casing.
Dropped 9 duplicate rows.
Verified: satisfaction_score remains NaN for all unresolved complaints (MNAR preserved).
PASS: plan_type has exactly 3 unique values
PASS: No duplicate rows
PASS: Total rows == 400 (409 raw - 9 duplicates)
PASS: No imputation on unresolved (resolved==False)
PASS: All resolved complaints have a satisfaction score
All tests passed – data layer is trustworthy.

--- Cleaned data summary ---
Shape: (400, 9)
Unique plan_type values: ['Broadband', 'Postpaid', 'Prepaid']
Duplicate rows remaining: 0
Missing satisfaction_score (unresolved): 106
Rows with resolved==False & satisfaction not null: 0
Rows with resolved==True & satisfaction null: 0


### Task 3 — NRA: Data Layer Impact (20 pts)
Number from Task 2's actual printed output. Reason = causal mechanism only, stop before
restating the fix. Action = specific, committed, names what happens next (Day 194 ML layer).


In [4]:
# TASK 3: NRA - Data Layer Impact
# Goal: document the cleaning impact using only values printed in Task 2's output.
# Method: retrieve numbers from live variables (df_raw, df_clean) and state causal mechanism.

nra_data_layer = {
    "number": f"df_raw had {df_raw['plan_type'].nunique()} plan_type values across {len(df_raw)} rows; "
              f"df_clean has {df_clean['plan_type'].nunique()} values across {len(df_clean)} rows.",
    "reason": "The casing variations (e.g., 'prepaid', 'POSTPAID', 'Broadband ') were deliberately injected to simulate inconsistent manual data entry across multiple support channels.",
    "action": "df_clean is the only input that Day 194's ML layer and Day 195's RAG layer are permitted to read; df_raw is never referenced again after this cell."
}

for k, v in nra_data_layer.items():
    print(f"{k.upper()}: {v}")

NUMBER: df_raw had 6 plan_type values across 409 rows; df_clean has 3 values across 400 rows.
REASON: The casing variations (e.g., 'prepaid', 'POSTPAID', 'Broadband ') were deliberately injected to simulate inconsistent manual data entry across multiple support channels.
ACTION: df_clean is the only input that Day 194's ML layer and Day 195's RAG layer are permitted to read; df_raw is never referenced again after this cell.


### Task 4 — Persist Layer Output + Repo Skeleton Note (15 pts)
The cleaned dataframe is the actual deliverable of today's layer — it must be written to disk,
not just held in memory, because Day 194's ML layer and Day 195's RAG layer load it fresh rather
than re-running today's notebook.


In [5]:
# TASK 4: Persist df_clean to disk + log the transformation summary
# Goal: write the cleaned data and a summary JSON so downstream layers can load them fresh.
# Method: use df_clean.to_csv and json.dump with values read from actual objects.

import json

output_path = "teleserve_clean.csv"
df_clean.to_csv(output_path, index=False)

transformation_log = {
    "source_rows": len(df_raw),
    "output_rows": len(df_clean),
    "duplicates_removed": len(df_raw) - len(df_clean),   # only row-count change is dedupe
    "plan_type_values_before": sorted(df_raw["plan_type"].unique().tolist()),
    "plan_type_values_after": sorted(df_clean["plan_type"].unique().tolist()),
    "satisfaction_score_missing_after": int(df_clean["satisfaction_score"].isna().sum()),
    "satisfaction_score_imputed": False,  # locked – per Day192 MNAR decision, never True
}

with open("transformation_log.json", "w") as f:
    json.dump(transformation_log, f, indent=2)

print("Transformation log written to transformation_log.json")
print(json.dumps(transformation_log, indent=2))

Transformation log written to transformation_log.json
{
  "source_rows": 409,
  "output_rows": 400,
  "duplicates_removed": 9,
  "plan_type_values_before": [
    "Broadband",
    "Broadband ",
    "POSTPAID",
    "Postpaid",
    "Prepaid",
    "prepaid"
  ],
  "plan_type_values_after": [
    "Broadband",
    "Postpaid",
    "Prepaid"
  ],
  "satisfaction_score_missing_after": 106,
  "satisfaction_score_imputed": false
}


**Repo skeleton note (for Day 196's README, not scored today):** this cell's two output files
(`teleserve_clean.csv`, `transformation_log.json`) belong in `Month12-Capstone-Netherlands-Portfolio/
data/processed/` — never in the same folder as raw data, and never committed if the CSV contains
anything sensitive (it doesn't here, but the habit is the point).


### Bonus (10★) — Before/After Comparison Table
One small dataframe, two rows (`before`, `after`), same columns as the transformation log's keys
that make sense as a table (row count, duplicate count, unique plan_type count). This is the
visual that goes straight into Day 196's final report without modification.


In [6]:
# BONUS: before/after comparison table
# Goal: provide a quick visual summary of the cleaning impact.
# Method: build a small DataFrame comparing df_raw vs df_clean on key metrics.

comparison = pd.DataFrame({
    "Metric": ["row_count", "duplicate_count", "unique_plan_type_count"],
    "Before": [len(df_raw), df_raw.duplicated().sum(), df_raw["plan_type"].nunique()],
    "After":  [len(df_clean), df_clean.duplicated().sum(), df_clean["plan_type"].nunique()]
})
print(comparison)

                   Metric  Before  After
0               row_count     409    400
1         duplicate_count       9      0
2  unique_plan_type_count       6      3


---
## Section 3 — Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 — Pipeline class | 30 | All 6 methods correctly implemented; one operation per method; `normalize_plan_type` runs before `drop_duplicates` |
| Task 2 — Test suite | 20 | All 5 assertions present, each prints PASS/FAIL with actual check description (not a bare `assert`), raises if any fail |
| Task 3 — NRA | 20 | Number from actual Task 2 output only; Reason states the injection mechanism without restating the fix; Action names the specific downstream constraint (df_clean-only rule) |
| Task 4 — Persist + log | 15 | CSV and JSON both written to disk via code; `satisfaction_score_imputed` stays `False`; log values read from real objects, not hardcoded |
| Bonus — Comparison table | 10★ | Correct 2-row table, values match Task 2's printed output exactly |
| **Total** | **85 + 10★** | |

**Technical vs Communication split for this day:**
- **Technical failure examples:** dedupe run before casing normalization (misses casing-variant duplicates); `verify_satisfaction_untouched` never called in `run()`; hardcoded `400` instead of computed row count in the log.
- **Communication failure examples:** Reason restates "so now it's fixed" instead of stopping at the injection mechanism; Action says "will use clean data going forward" without naming which specific files/notebooks are bound by that rule.


---
## Interview Framing

**If asked: "Walk me through how you'd clean a messy production dataset before feeding it to a
model."**

> "I never clean in place. I built a pipeline class with one method per operation — normalize
> categorical casing, drop duplicates, and a *verification* method, not a fix, for a field that
> was structurally missing rather than randomly missing. Order matters: I normalize casing before
> deduplication, because two rows that differ only by casing on a category column will slip past
> `.duplicated()` if you dedupe first. Every method is gated by an assertion test suite that prints
> pass/fail per check and raises if anything fails, so a broken cleaning step can never silently
> hand bad data to the ML or reporting layer downstream. And critically — I don't impute everything
> that's missing. If a value is missing *because* of what happened in the business process, like a
> satisfaction score that only exists for resolved complaints, imputing it manufactures an opinion
> that was never given. I leave it as null and let the next layer decide how to handle that,
> explicitly."

**Key Takeaway:** A cleaning pipeline's job is not to remove every irregularity — it's to remove
the irregularities that are *artifacts of bad entry* (casing, duplicates) while explicitly
preserving the irregularities that are *signal about the business process* (structurally missing
satisfaction scores). Conflating the two is the single most common way a "cleaned" dataset ends up
lying to the model trained on it.
